# From Linguistic Context to Semantic Structure — Tests

Or Shulrufer

**Purpose — test notebook.** This optional notebook checks the actual computational notebook. It reads that notebook's source cells and exercises their functions without copying their implementation. It uses explicitly constructed unit-test inputs, which are not corpus samples or classification evidence. The computational experiment runs in GitHub Actions.

The current tests cover the implemented acquisition and normalization stage. This notebook will gain stage-specific tests as chunking, numerical operations and classification are connected to the same computational notebook. Passing these tests does not establish full experiment completion or classification performance.

The figures notebook defines manuscript illustrations without rendering before approval. Unicode Markdown and LaTeX are the two manuscript sources. After all notebooks are validated, an optional single Python pipeline will be derived for deployment through suitable scikit-learn and PyTorch interfaces.

In [1]:
# Purpose: use the computational notebook's one dependency specification.
from pathlib import Path
import json
project = Path.cwd().resolve()
if (project / "Paper_Final").is_dir():
    project = project / "Paper_Final"
computational_notebook = json.loads((project / "Proof.ipynb").read_text())
dependency_cells = [cell for cell in computational_notebook["cells"] if cell.get("id") == "dependencies"]
assert len(dependency_cells) == 1
exec(compile("".join(dependency_cells[0]["source"]), "computational-notebook-dependencies", "exec"))


In [2]:
# Purpose: test the canonical computational notebook without maintaining a second implementation.
from pathlib import Path
import ast
import hashlib
import json
import tempfile
import unittest
from unittest.mock import patch
import io
import math
from datetime import datetime, timezone, timedelta
from email.utils import parsedate_to_datetime, format_datetime
from threading import Lock
from collections import Counter
import unicodedata
import urllib.request
import urllib.error
import time
import nbformat

PROJECT_DIRECTORY = Path.cwd().resolve()
if (PROJECT_DIRECTORY / "Paper_Final").is_dir():
    PROJECT_DIRECTORY = PROJECT_DIRECTORY / "Paper_Final"
COMPUTATION_PATH = PROJECT_DIRECTORY / "Proof.ipynb"
FIGURES_PATH = PROJECT_DIRECTORY / "Figure_Generator.ipynb"
computation = nbformat.read(COMPUTATION_PATH, as_version=4)
figures = nbformat.read(FIGURES_PATH, as_version=4)

def actual_function(notebook, name, namespace):
    matches = []
    for cell in notebook.cells:
        if cell.cell_type == "code":
            matches.extend(node for node in ast.parse(cell.source).body
                           if isinstance(node, ast.FunctionDef) and node.name == name)
    assert len(matches) == 1, f"Expected one canonical definition of {name}"
    exec(compile(ast.Module(body=matches, type_ignores=[]), str(COMPUTATION_PATH), "exec"), namespace)
    return namespace[name]

namespace = {"Counter": Counter, "unicodedata": unicodedata, "hashlib": hashlib,
             "urllib": urllib, "time": time, "json": json,
             "math": math, "datetime": datetime, "timezone": timezone,
             "parsedate_to_datetime": parsedate_to_datetime,
             "SOURCE_REQUEST_LOCK": Lock(), "NEXT_SOURCE_REQUEST_AT": [0.0],
             "SOURCE_REQUEST_INTERVAL": 0.2}
normalize_text = actual_function(computation, "normalize_text", namespace)
verify_source_bytes = actual_function(computation, "verify_source_bytes", namespace)
wait_for_source_request = actual_function(computation, "wait_for_source_request", namespace)
defer_source_requests = actual_function(computation, "defer_source_requests", namespace)
fetch_source_bytes = actual_function(computation, "fetch_source_bytes", namespace)
obtain_existing_text = actual_function(computation, "obtain_existing_text", namespace)

class ComputationalNotebookTests(unittest.TestCase):
    def test_notebooks_have_valid_unique_cells(self):
        for notebook in (computation, figures):
            nbformat.validate(notebook)
            self.assertEqual(len({cell.id for cell in notebook.cells}), len(notebook.cells))
            for index, cell in enumerate(notebook.cells):
                if cell.cell_type == "code":
                    compile(cell.source, f"notebook-cell-{index}", "exec")

    def test_character_normalization(self):
        text, edits, counts = normalize_text("Hello\r\nWORLD\tA\u00a0B\x00")
        self.assertEqual(text, "hello\nworld a b")
        self.assertTrue(edits)
        self.assertEqual(counts["nonprinting_removed"], 1)
        self.assertEqual(normalize_text(text)[0], text)

    def test_unicode_lowercase_expansion_is_recorded(self):
        text, edits, counts = normalize_text("İ")
        self.assertEqual(text, "i\u0307")
        self.assertEqual(edits[0]["source_end"], 1)
        self.assertEqual(edits[0]["normalized_end"], 2)
        self.assertEqual(counts["lowercase_character_expansion"], 1)

    def test_empty_broken_and_long_texts_remain(self):
        self.assertEqual(normalize_text("")[0], "")
        self.assertEqual(normalize_text("Words WITHOUT sentence punctuation")[0], "words without sentence punctuation")
        raw = "Mixed text without a final sentence. " * 100000
        self.assertEqual(normalize_text(raw)[0], raw.lower())
        self.assertEqual(len(normalize_text(raw)[0]), len(raw))

    def test_original_bytes_require_complete_length_and_hash(self):
        data = b"Existing extracted text\n"
        asset = {"path": "2001.00001/full.md", "bytes": len(data), "sha256": None,
                 "git_blob_sha1": hashlib.sha1(f"blob {len(data)}\0".encode() + data).hexdigest()}
        verify_source_bytes(data, asset)
        with self.assertRaises(AssertionError):
            verify_source_bytes(data[:-1], asset)
        with self.assertRaises(AssertionError):
            verify_source_bytes(b"X" + data[1:], asset)
        asset["sha256"] = hashlib.sha256(data).hexdigest()
        verify_source_bytes(data, asset)

    def test_resume_reuses_verified_bytes_without_network(self):
        data = b"Already downloaded complete text"
        asset = {"path": "2001.00001/full.md", "bytes": len(data),
                 "sha256": hashlib.sha256(data).hexdigest(), "git_blob_sha1": None}
        record = {"document_id": "2001.00001", "assets": [asset]}
        with tempfile.TemporaryDirectory() as directory:
            directory = Path(directory)
            (directory / "2001.00001.md").write_bytes(data)
            namespace["RAW_TEXT_DIRECTORY"] = directory
            with patch("urllib.request.urlopen", side_effect=AssertionError("Resume unexpectedly used the network")):
                result = obtain_existing_text(record)
            self.assertEqual(result["bytes"], len(data))
            self.assertEqual((directory / "2001.00001.md").read_bytes(), data)

    def test_shared_request_schedule_spaces_requests(self):
        clock = [100.0]
        namespace["NEXT_SOURCE_REQUEST_AT"] = [100.0]
        with patch("time.monotonic", side_effect=lambda: clock[0]), patch("time.sleep", side_effect=lambda delay: clock.__setitem__(0, clock[0] + delay)):
            wait_for_source_request()
            wait_for_source_request()
            wait_for_source_request()
        self.assertAlmostEqual(clock[0], 100.4)

    def test_rate_limit_pauses_all_workers_and_honors_retry_after(self):
        namespace["NEXT_SOURCE_REQUEST_AT"] = [0.0]
        with patch("time.monotonic", return_value=100.0):
            delay = defer_source_requests({"Retry-After": "120"}, 0, 429)
            self.assertEqual(delay, 120.0)
            self.assertEqual(namespace["NEXT_SOURCE_REQUEST_AT"][0], 220.0)
            defer_source_requests({}, 0, 503)
            self.assertEqual(namespace["NEXT_SOURCE_REQUEST_AT"][0], 220.0)

    def test_retry_after_http_date_and_malformed_header(self):
        future = datetime.now(timezone.utc) + timedelta(seconds=120)
        delay = defer_source_requests({"Retry-After": format_datetime(future, usegmt=True)}, 0, 429)
        self.assertGreater(delay, 115)
        self.assertLessEqual(delay, 120)
        self.assertEqual(defer_source_requests({"Retry-After": "invalid"}, 0, 429), 60.0)

    def test_rate_limited_download_retries_and_returns_complete_bytes(self):
        error = urllib.error.HTTPError("https://example.invalid", 429, "Too Many Requests", {"Retry-After": "120"}, None)
        response = io.BytesIO(b"complete source bytes")
        with patch.dict(namespace, {"wait_for_source_request": lambda: None}), patch("urllib.request.urlopen", side_effect=[error, response]) as opened:
            with patch.dict(namespace, {"defer_source_requests": lambda headers, attempt, status: self.assertEqual((headers["Retry-After"], attempt, status), ("120", 0, 429))}):
                self.assertEqual(fetch_source_bytes("https://example.invalid"), b"complete source bytes")
            self.assertEqual(opened.call_count, 2)

    def test_permanent_http_error_is_not_retried(self):
        error = urllib.error.HTTPError("https://example.invalid", 404, "Not Found", {}, None)
        with patch.dict(namespace, {"wait_for_source_request": lambda: None}), patch("urllib.request.urlopen", side_effect=error) as opened:
            with self.assertRaises(urllib.error.HTTPError):
                fetch_source_bytes("https://example.invalid")
            self.assertEqual(opened.call_count, 1)

    def test_repeated_rate_limit_preserves_failure_for_resumable_run(self):
        error = urllib.error.HTTPError("https://example.invalid", 429, "Too Many Requests", {}, None)
        with patch.dict(namespace, {"wait_for_source_request": lambda: None, "defer_source_requests": lambda *args: None}), patch("urllib.request.urlopen", side_effect=error) as opened:
            with self.assertRaises(urllib.error.HTTPError):
                fetch_source_bytes("https://example.invalid")
            self.assertEqual(opened.call_count, 8)

    def test_figure_rendering_requires_verified_complete_results(self):
        with tempfile.TemporaryDirectory() as directory:
            scope={"RESULTS_DIRECTORY":Path(directory),"DATASET_REVISION":"expected", "json":json}
            guard=actual_function(figures,"require_verified_results",scope)
            path=Path(directory)/"verification.json"
            path.write_text(json.dumps(dict(complete=False,dataset_revision="expected",source_documents=8836)))
            with self.assertRaises(AssertionError): guard()
            path.write_text(json.dumps(dict(complete=True,dataset_revision="wrong",source_documents=8836)))
            with self.assertRaises(AssertionError): guard()
            path.write_text(json.dumps(dict(complete=True,dataset_revision="expected",source_documents=8836)))
            self.assertTrue(guard()["complete"])

suite = unittest.defaultTestLoader.loadTestsFromTestCase(ComputationalNotebookTests)
test_result = unittest.TextTestRunner(verbosity=2).run(suite)
assert test_result.wasSuccessful(), "Revise the computational notebook and rerun the affected tests."
print(json.dumps({"scope": "Implemented acquisition and normalization functions; figure-rendering guard",
                  "tests_run": test_result.testsRun, "successful": test_result.wasSuccessful(),
                  "experiment_complete": False,
                  "computational_notebook_sha256": hashlib.sha256(COMPUTATION_PATH.read_bytes()).hexdigest()}, indent=2))


test_character_normalization (__main__.ComputationalNotebookTests.test_character_normalization) ... ok
test_empty_broken_and_long_texts_remain (__main__.ComputationalNotebookTests.test_empty_broken_and_long_texts_remain) ... ok
test_figure_rendering_requires_verified_complete_results (__main__.ComputationalNotebookTests.test_figure_rendering_requires_verified_complete_results) ... ok
test_notebooks_have_valid_unique_cells (__main__.ComputationalNotebookTests.test_notebooks_have_valid_unique_cells) ... ok
test_original_bytes_require_complete_length_and_hash (__main__.ComputationalNotebookTests.test_original_bytes_require_complete_length_and_hash) ... ok
test_permanent_http_error_is_not_retried (__main__.ComputationalNotebookTests.test_permanent_http_error_is_not_retried) ... ok
test_rate_limit_pauses_all_workers_and_honors_retry_after (__main__.ComputationalNotebookTests.test_rate_limit_pauses_all_workers_and_honors_retry_after) ... ok
test_rate_limited_download_retries_and_returns_comp

## Scientific implementation tests

These tests import the actual definitions from the proof notebook. Constructed fixtures verify numerical identities, source retention, checkpoint integrity and the feature-dependent BERT gradient path. They are separate from full-corpus measurements.

In [3]:

# Purpose: exercise the same mathematical, source-partition and BERT training definitions as Proof.ipynb.
import csv, gc, types, inspect
import numpy as np
from scipy import sparse
import matplotlib.pyplot as plt
scientific = dict(__name__='__main__',Path=Path,json=json,hashlib=hashlib,np=np,sparse=sparse,
                  Counter=Counter,csv=csv,plt=plt,PROJECT_DIRECTORY=PROJECT_DIRECTORY,
                  DATASET_REVISION='a68009864dbc3b1decb1a37a6c4f769887169a44')
scientific['save_json']=actual_function(computation,'save_json',dict(Path=Path,json=json))
for cell in computation.cells:
    if cell.metadata.get('definition_cell'):
        exec(compile(cell.source,str(COMPUTATION_PATH)+':'+cell.id,'exec'),scientific)
scientific['RUN_BUDGET']=0

def make_bert(hidden_size=24,layers=2):
    return scientific['BertModel'](scientific['BertConfig'](vocab_size=60,hidden_size=hidden_size,
        num_hidden_layers=layers,num_attention_heads=4,intermediate_size=48,
        hidden_dropout_prob=0.,attention_probs_dropout_prob=0.,max_position_embeddings=512),add_pooling_layer=False)

class ScientificPipelineTests(unittest.TestCase):
    def test_numerical_reference_against_independent_calculations(self):
        report=scientific['numerical_self_test']()
        self.assertEqual(report['status'],'PASS')
        self.assertGreaterEqual(len(report['checks']),17)

    def test_separator_ends_cover_irregular_text_exactly(self):
        text='first without punctuation\nsecond broken table | 3 |\nlast'
        predictions=[dict(start=5,end=8),dict(start=20,end=24)]
        ends=scientific['validated_chunk_ends'](text,predictions)
        self.assertEqual(ends,[8,24,len(text)])
        self.assertEqual(''.join(text[a:b] for a,b in zip([0]+ends[:-1],ends)),text)
        self.assertEqual(scientific['validated_chunk_ends']('',[]),[0])
        self.assertEqual(scientific['validated_chunk_ends'](text,[]),[len(text)])
        with self.assertRaises(AssertionError):
            scientific['validated_chunk_ends'](text,list(reversed(predictions)))

    def test_partitions_retain_all_sources_and_join_duplicate_families(self):
        records=[]; meta={}
        for i in range(31):
            identifier=f'source-{i:03d}'
            records.append(dict(document_id=identifier,source_sha256=str(i),normalized_sha256=str(i//2) if i<4 else str(i)))
            meta[identifier]=dict(primary_category='rare' if i==30 else ('a' if i%2==0 else 'b'),doi='same-doi' if i in (7,9) else '')
        a=scientific['source_partitions'](records,meta); b=scientific['source_partitions'](records,meta)
        self.assertEqual(a,b); self.assertEqual(len(a['documents']),31)
        docs=a['documents']; self.assertEqual(docs[0]['split'],docs[1]['split']); self.assertEqual(docs[7]['split'],docs[9]['split'])
        self.assertEqual(docs[-1]['split'],'train')
        self.assertTrue(all(count>0 for count in a['class_support']['train'].values()))

    def test_large_artifacts_roundtrip_and_corruption_rejection(self):
        with tempfile.TemporaryDirectory() as directory:
            p=Path(directory)/'weights.bin'; original=bytes(range(256))*8; p.write_bytes(original)
            scientific['pack_large_files'](directory,threshold=100,part_size=137)
            self.assertEqual(p.read_bytes(),original)
            p.unlink(); scientific['restore_large_files'](directory)
            self.assertEqual(p.read_bytes(),original)
            p.unlink(); (Path(directory)/'weights.bin.parts/000000.bin').write_bytes(b'broken')
            with self.assertRaises(AssertionError): scientific['restore_large_files'](directory)

    def test_statistical_features_affect_training_gradients(self):
        torch=scientific['torch']; torch.manual_seed(29)
        model=scientific['ContextualBehaviorBert'](make_bert(),3,2)
        ids=torch.tensor([[2,6,7,3],[2,8,9,3]])
        attention=torch.ones_like(ids); target=torch.tensor([[0,1,0,0],[0,0,1,0]])
        features=torch.tensor([[1.,2.,3.],[3.,1.,2.]])
        labels=torch.tensor([0,1]); last=next(model.bert.encoder.layer[-1].parameters())
        first=next(model.bert.encoder.layer[0].parameters()); before_first=first.detach().clone(); before_last=last.detach().clone()
        output=model(ids,attention,target,features,labels)
        torch.testing.assert_close(output['loss'],torch.nn.functional.cross_entropy(output['logits'],labels,reduction='mean'))
        output['loss'].backward(); original_gradient=last.grad.detach().clone()
        self.assertGreater(float(model.head[0].weight.grad[:,24:].norm()),0)
        model.zero_grad(set_to_none=True); model(ids,attention,target,features*2,labels)['loss'].backward()
        self.assertFalse(torch.equal(original_gradient,last.grad))
        optimizer=torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],lr=.001)
        optimizer.step()
        self.assertFalse(torch.equal(before_last,last)); self.assertTrue(torch.equal(before_first,first))
        self.assertFalse(first.requires_grad); self.assertFalse(model.bert.embeddings.word_embeddings.weight.requires_grad)
        self.assertFalse(model.bert.encoder.layer[0].training); self.assertTrue(model.bert.encoder.layer[-1].training)

    def test_bert_baseline_accepts_no_auxiliary_features(self):
        torch=scientific['torch']
        model=scientific['ContextualBehaviorBert'](make_bert(),0,2)
        ids=torch.tensor([[2,6,3]]); mask=torch.ones_like(ids); target=torch.tensor([[0,1,0]])
        result=model(ids,mask,target,torch.empty(1,0),torch.tensor([0]))
        self.assertEqual(tuple(result['logits'].shape),(1,2)); self.assertTrue(torch.isfinite(result['loss']))
        with self.assertRaises(ValueError): model(ids,mask,torch.zeros_like(ids),torch.empty(1,0))
        with self.assertRaises(ValueError): model(ids.repeat(1,172),mask.repeat(1,172),target.repeat(1,172),torch.empty(1,0))

    def test_frozen_controls_and_checkpoint_restore(self):
        torch=scientific['torch']; torch.manual_seed(8)
        model=scientific['FrozenPassageBert'](make_bert(),4,2,3,'legacy_frozen_svd_cv')
        self.assertFalse(any(p.requires_grad for p in model.bert.parameters()))
        saved=scientific['model_trainable_state'](model)
        with torch.no_grad(): model.head[0].weight.add_(1)
        scientific['restore_trainable_state'](model,saved)
        torch.testing.assert_close(model.head[0].weight,saved['head.0.weight'])
        self.assertFalse(model.bert.training)

    def test_source_aggregation_retains_unavailable_targets(self):
        corpus=dict(topics=['a','b'],documents=[dict(id=0,label=0,split='train'),dict(id=1,label=1,split='train'),
                   dict(id=2,label=1,split='test'),dict(id=3,label=0,split='test')],
                   targets=[dict(doc=0,label=0,split='train'),dict(doc=1,label=1,split='train'),
                            dict(doc=2,label=1,split='test'),dict(doc=3,label=0,split='test')])
        records=[dict(has_target=True),dict(has_target=True),dict(has_target=True),dict(has_target=False)]
        population=scientific['Population'](corpus,2,types.SimpleNamespace(records=records))
        ids,y,p=population.aggregate(np.array([[.1,.9]]),np.array([2]),'test')
        self.assertEqual(ids,[2,3]); np.testing.assert_allclose(p,[[.1,.9],[.5,.5]])
        self.assertEqual(population.fallback['test'],[3])

    def test_paired_bootstrap_identical_models_have_zero_difference(self):
        predictions=np.array([[0,1,0,1],[1,1,0,0],[0,1,1,0]])
        report=scientific['paired_bootstrap'](np.array([0,1,1,0]),{'left':predictions,'right':predictions.copy()},
            ['a','a','b','c'],2,repetitions=100,pairs=(('left','right'),))
        for value in report['comparisons'][0]['statistics'].values():
            self.assertEqual(value['mean_seed_difference'],0); self.assertEqual(value['percentile_95_interval'],[0.,0.])


    def test_complete_99_candidate_numerical_pipeline_and_resume(self):
        import contextlib
        with tempfile.TemporaryDirectory() as directory:
            out=Path(directory)
            old=scientific.get('RESULTS_DIRECTORY'); scientific['RESULTS_DIRECTORY']=out
            scientific['display_figure']=lambda figure,name:plt.close(figure)
            (out/'segmentation.json').write_text(json.dumps({'identity':'fixture-segments'}))
            (out/'source_partitions.json').write_text(json.dumps({'identity':'fixture-partitions'}))
            corpus=dict(documents=[],snippets=[],topics=['a','b']); rng=np.random.default_rng(42)
            vocabulary=['word'+str(i) for i in range(45)]
            for i in range(150):
                text=' '.join(rng.choice(vocabulary,15).tolist())
                split='train' if i<120 else ('validation' if i<135 else 'test')
                if split!='train': text+=' heldoutonly'
                corpus['documents'].append(dict(id=i,document_id=str(i),label=i%2,split=split,family=i,text=text,body_start=0))
                corpus['snippets'].append(dict(doc=i,label=i%2,split=split,start=0,end=len(text),text=text))
            with contextlib.redirect_stdout(io.StringIO()):
                report=scientific['fit_numerical_reference'](corpus)
                repeated=scientific['fit_numerical_reference'](corpus)
            self.assertEqual(report,repeated)
            self.assertTrue(report['checks']['all_99_candidates_completed'])
            self.assertNotIn('heldoutonly',json.loads((out/'vocabulary.json').read_text())['words'])
            self.assertEqual(len(corpus['targets']),150)
            scientific['RESULTS_DIRECTORY']=old

suite=unittest.defaultTestLoader.loadTestsFromTestCase(ScientificPipelineTests)
scientific_result=unittest.TextTestRunner(verbosity=2).run(suite)
assert scientific_result.wasSuccessful(),'Repair the actual computational notebook before execution.'
print('Scientific integration checks passed. Fixtures establish mechanics, not corpus classification performance.')


test_bert_baseline_accepts_no_auxiliary_features (__main__.ScientificPipelineTests.test_bert_baseline_accepts_no_auxiliary_features) ... ok
test_complete_99_candidate_numerical_pipeline_and_resume (__main__.ScientificPipelineTests.test_complete_99_candidate_numerical_pipeline_and_resume) ... ok
test_frozen_controls_and_checkpoint_restore (__main__.ScientificPipelineTests.test_frozen_controls_and_checkpoint_restore) ... ok
test_large_artifacts_roundtrip_and_corruption_rejection (__main__.ScientificPipelineTests.test_large_artifacts_roundtrip_and_corruption_rejection) ... ok
test_numerical_reference_against_independent_calculations (__main__.ScientificPipelineTests.test_numerical_reference_against_independent_calculations) ... ok
test_paired_bootstrap_identical_models_have_zero_difference (__main__.ScientificPipelineTests.test_paired_bootstrap_identical_models_have_zero_difference) ... ok
test_partitions_retain_all_sources_and_join_duplicate_families (__main__.ScientificPipelineTests.tes

In [4]:

# Purpose: verify the actual downloaded pretrained models, not classification performance on test fixtures.
import os
TEST_MODEL_ROOT=Path(os.environ.get('PRETRAINED_TEST_MODEL_DIRECTORY',str(PROJECT_DIRECTORY/'corpus/models')))
if 'PRETRAINED_TEST_MODEL_DIRECTORY' in os.environ:
    # A local testing override reuses the already downloaded checkpoint; the GitHub path obtains exact pinned revisions.
    BERT_TEST_DIRECTORY=TEST_MODEL_ROOT/'bert'
    SEGMENTER_TEST_DIRECTORY=TEST_MODEL_ROOT/'context_segmenter'
    assert scientific['file_sha256'](BERT_TEST_DIRECTORY/'model.safetensors')=='68d45e234eb4a928074dfd868cead0219ab85354cc53d20e772753c6bb9169d3'
    assert json.loads((SEGMENTER_TEST_DIRECTORY/'download_manifest.json').read_text())['revision']==scientific['SEGMENTER_REVISION']
else:
    BERT_TEST_DIRECTORY=scientific['obtain_pretrained_model'](scientific['BERT_REPOSITORY'],scientific['BERT_REVISION'],TEST_MODEL_ROOT/'bert')
    SEGMENTER_TEST_DIRECTORY=scientific['obtain_pretrained_model'](scientific['SEGMENTER_REPOSITORY'],scientific['SEGMENTER_REVISION'],TEST_MODEL_ROOT/'context_segmenter')

class PretrainedInterfaceTests(unittest.TestCase):
    def test_actual_pretrained_bert_alignment_and_update(self):
        torch=scientific['torch']; torch.manual_seed(29)
        tokenizer=scientific['BertTokenizerFast'].from_pretrained(BERT_TEST_DIRECTORY,local_files_only=True)
        text='medical research explores tissue structure'
        data=dict(snippets=[dict(text=text)],targets=[dict(snippet=0,start=8,end=16,has_target=True)])
        records,_,_=scientific['target_windows'](data,tokenizer,512)
        batch,target=scientific['token_batch'](records,[0])
        bert=scientific['BertModel'].from_pretrained(BERT_TEST_DIRECTORY,local_files_only=True,add_pooling_layer=False,attn_implementation='sdpa')
        self.assertEqual(bert.config.hidden_size,768); self.assertEqual(len(bert.encoder.layer),12)
        model=scientific['ContextualBehaviorBert'](bert,3,2)
        last=next(bert.encoder.layer[-1].parameters()); before=last.detach().clone()
        output=model(**batch,target_mask=target,features=torch.tensor([[.2,.3,.4]]),labels=torch.tensor([1]))
        self.assertTrue(torch.isfinite(output['loss'])); output['loss'].backward()
        self.assertGreater(float(last.grad.norm()),0)
        optimizer=torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],lr=.00002);optimizer.step()
        self.assertFalse(torch.equal(before,last))
        del optimizer,model,bert; gc.collect()

    def test_actual_context_model_and_original_boundary_rule(self):
        tokenizer=scientific['AutoTokenizer'].from_pretrained(SEGMENTER_TEST_DIRECTORY,model_max_length=1024,local_files_only=True)
        model=scientific['AutoModelForTokenClassification'].from_pretrained(SEGMENTER_TEST_DIRECTORY,local_files_only=True,attn_implementation='sdpa')
        self.assertEqual(model.config.id2label,{0:'O',1:'separator'})
        splitter=scientific['pipeline']('ner',model=model,tokenizer=tokenizer,device='cpu',aggregation_strategy='simple',stride=512)
        text='medical research without punctuation\ncell growth tissue structure\n| sample | measurement |\nresults need context'
        predictions=splitter(text); ends=scientific['validated_chunk_ends'](text,predictions)
        self.assertEqual(''.join(text[a:b] for a,b in zip([0]+ends[:-1],ends)),text)
        del splitter,model;gc.collect()

pretrained_result=unittest.TextTestRunner(verbosity=2).run(unittest.defaultTestLoader.loadTestsFromTestCase(PretrainedInterfaceTests))
assert pretrained_result.wasSuccessful()
print(json.dumps(dict(scope='Acquisition, numerical integration and real pretrained-model interfaces',
    tests_run=test_result.testsRun+scientific_result.testsRun+pretrained_result.testsRun,
    successful=True,experiment_complete=False,
    computational_notebook_sha256=hashlib.sha256(COMPUTATION_PATH.read_bytes()).hexdigest()),indent=2))


test_actual_context_model_and_original_boundary_rule (__main__.PretrainedInterfaceTests.test_actual_context_model_and_original_boundary_rule) ... ok
test_actual_pretrained_bert_alignment_and_update (__main__.PretrainedInterfaceTests.test_actual_pretrained_bert_alignment_and_update) ... ok

----------------------------------------------------------------------
Ran 2 tests in 5.609s

OK
{
  "scope": "Acquisition, numerical integration and real pretrained-model interfaces",
  "tests_run": 25,
  "successful": true,
  "experiment_complete": false,
  "computational_notebook_sha256": "5e237c12d09a5de67b61e472fefbab548298695659efb1000b89b1638cab4dd5"
}
